# Notebook 02 — Deterministic Modelling and Ensemble Evaluation

This notebook develops leakage-safe deterministic wind-generation forecasts for DK1 and DK2.

## Scope

1. Reload and verify the frozen outputs from Notebook 01.
2. Build a decision-time feature registry.
3. Define expanding, time-aware train-validation-test folds.
4. Examine target relationships, cross-provider dependence, residual correlation, and stability.
5. Engineer physically meaningful and decision-time-safe features.
6. Fit deterministic baselines.
7. Train provider-specific LightGBM experts.
8. Estimate static and dynamic ensemble weights.
9. Fit a pooled LightGBM benchmark.
10. Evaluate accuracy, bias, robustness, stability, and regime performance.
11. Preserve the long-history D2 and exact-run tracks as distinct experiments.

The notebook defaults to `RUN_MODE = "AUDIT"`. This runs the data, leakage, split, and exploratory gates without launching the full model search. Change it to `"FULL"` only after the audit status is satisfactory.

## Methodological rules

- No random train-test split.
- No feature may use information published after the forecast decision time.
- Correlation analysis and feature selection use training data only.
- Hyperparameters and ensemble weights are chosen on validation data only.
- The final test windows remain untouched until all modelling choices for that fold are fixed.
- DK1 and DK2 are evaluated separately.
- D2 Previous Runs and exact-run panels remain separate modelling tracks.
- The strict seven-provider exact-run panel is the canonical exact-run comparison dataset.
- DST is handled through timezone-aware timestamps and observed timestamp keys rather than assuming 24 observations per local day.

In [ ]:
# ============================================================
# 1. PROJECT SETUP
# ============================================================

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Sequence
import hashlib
import json
import math
import platform
import re
import sys
import warnings

import numpy as np
import pandas as pd

from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

RUN_MODE = "AUDIT"  # "AUDIT" or "FULL"
STRICT_SHAPE_CHECK = True

LOCAL_TIMEZONE = "Europe/Copenhagen"
PRICE_AREAS = ("DK1", "DK2")
TARGET_COLUMN = "actual_wind_mwh"
KEY_COLUMNS = ("timestamp_utc", "price_area")

D2_SOURCES = (
    "gfs",
    "icon_eu",
    "ecmwf",
    "gem_global",
    "jma_gsm",
)

EXACT_SOURCES = (
    "gfs",
    "icon_eu",
    "ecmwf",
    "dmi",
    "arpege_europe",
    "ukmo_global",
    "cma_grapes",
)

EXPECTED_D2_SHAPE = (33_648, 228)
EXPECTED_EXACT_SHAPE = (9_458, 315)

# Rolling-fold design. Adjust only before reviewing test results.
D2_FOLD_CONFIG = {
    "min_train_days": 365,
    "validation_days": 60,
    "test_days": 90,
    "step_days": 90,
}

EXACT_FOLD_CONFIG = {
    "min_train_days": 90,
    "validation_days": 30,
    "test_days": 45,
    "step_days": 30,
}

LIGHTGBM_BASE_PARAMS = {
    "objective": "regression_l1",
    "n_estimators": 2_000,
    "learning_rate": 0.025,
    "num_leaves": 31,
    "max_depth": -1,
    "min_child_samples": 40,
    "subsample": 0.9,
    "colsample_bytree": 0.9,
    "reg_alpha": 0.0,
    "reg_lambda": 1.0,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
    "verbosity": -1,
}

print(
    {
        "run_mode": RUN_MODE,
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "random_seed": RANDOM_SEED,
    }
)

In [ ]:
# ============================================================
# 2. LOCATE PROJECT ROOT AND OUTPUT DIRECTORIES
# ============================================================

def find_project_root(starting_path: Path | None = None) -> Path:
    current = Path(starting_path or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (
            candidate.joinpath("pyproject.toml").exists()
            and candidate.joinpath("data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Expected pyproject.toml and data/."
    )


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

OUTPUT_DIR = PROJECT_ROOT / "outputs"
AUDIT_DIR = OUTPUT_DIR / "audits"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
PREDICTION_DIR = OUTPUT_DIR / "predictions"
TABLE_DIR = OUTPUT_DIR / "tables"

for directory in (
    AUDIT_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    PREDICTION_DIR,
    TABLE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

## Frozen input contract

Notebook 02 does not reconstruct data. It consumes the canonical Parquet outputs from Notebook 01.

Because earlier revisions used more than one historical-panel filename, the resolver below checks an ordered list of candidates. The exact-run canonical filename should resolve to the strict seven-provider common panel.

In [ ]:
# ============================================================
# 3. RESOLVE CANONICAL INPUT FILES
# ============================================================

D2_PANEL_CANDIDATES = (
    PROCESSED_DIR / "wind_model_panel_d2_hourly.parquet",
    PROCESSED_DIR / "wind_model_panel_hourly.parquet",
    PROCESSED_DIR / "wind_model_panel.parquet",
    PROCESSED_DIR / "wind_expert_model_panel.parquet",
)

EXACT_PANEL_CANDIDATES = (
    PROCESSED_DIR / "wind_model_panel_exact_runs_hourly.parquet",
    PROCESSED_DIR / "wind_model_panel_exact_runs_seven_source_common_hourly.parquet",
)


def resolve_existing_path(
    candidates: Sequence[Path],
    *,
    label: str,
) -> Path:
    existing = [path for path in candidates if path.exists()]

    if not existing:
        raise FileNotFoundError(
            f"No {label} file was found. Checked: "
            + ", ".join(str(path) for path in candidates)
        )

    if len(existing) > 1:
        print(
            f"Multiple {label} files exist. Using the first candidate:",
            existing[0],
        )

    return existing[0]


D2_PANEL_PATH = resolve_existing_path(
    D2_PANEL_CANDIDATES,
    label="D2 panel",
)

EXACT_PANEL_PATH = resolve_existing_path(
    EXACT_PANEL_CANDIDATES,
    label="exact-run panel",
)

print("D2 panel:", D2_PANEL_PATH)
print("Exact-run panel:", EXACT_PANEL_PATH)

In [ ]:
# ============================================================
# 4. LOAD AND VALIDATE FROZEN PANELS
# ============================================================

def sha256_file(
    path: Path,
    chunk_size: int = 1_048_576,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)

    return digest.hexdigest()


def validate_model_panel(
    frame: pd.DataFrame,
    *,
    panel_name: str,
    expected_shape: tuple[int, int],
) -> pd.DataFrame:
    required_columns = {
        "timestamp_utc",
        "price_area",
        TARGET_COLUMN,
    }

    missing_columns = sorted(required_columns.difference(frame.columns))

    if missing_columns:
        raise RuntimeError(
            f"{panel_name} is missing required columns: {missing_columns}"
        )

    if STRICT_SHAPE_CHECK and frame.shape != expected_shape:
        raise RuntimeError(
            f"{panel_name} shape mismatch: "
            f"observed={frame.shape}, expected={expected_shape}"
        )

    validated = frame.copy()

    timestamps = pd.to_datetime(
        validated["timestamp_utc"],
        errors="raise",
    )

    if timestamps.dt.tz is None:
        raise RuntimeError(
            f"{panel_name} contains timezone-naive timestamps."
        )

    validated["timestamp_utc"] = timestamps.dt.tz_convert("UTC")

    # Force the modelling target to numeric at the dataset gate.
    # Any non-numeric value becomes missing and is rejected below.
    validated[TARGET_COLUMN] = pd.to_numeric(
        validated[TARGET_COLUMN],
        errors="coerce",
    )

    duplicate_count = int(
        validated.duplicated(list(KEY_COLUMNS)).sum()
    )

    if duplicate_count:
        raise RuntimeError(
            f"{panel_name} contains {duplicate_count} duplicate keys."
        )

    observed_areas = set(
        validated["price_area"].dropna().astype(str).unique()
    )

    if observed_areas != set(PRICE_AREAS):
        raise RuntimeError(
            f"{panel_name} price-area mismatch: "
            f"observed={sorted(observed_areas)}, "
            f"expected={sorted(PRICE_AREAS)}"
        )

    missing_target_count = int(
        validated[TARGET_COLUMN].isna().sum()
    )

    if missing_target_count:
        raise RuntimeError(
            f"{panel_name} has {missing_target_count} "
            "missing or non-numeric target values."
        )

    return (
        validated.sort_values(list(KEY_COLUMNS))
        .reset_index(drop=True)
    )


d2_panel = validate_model_panel(
    pd.read_parquet(D2_PANEL_PATH),
    panel_name="D2 panel",
    expected_shape=EXPECTED_D2_SHAPE,
)

exact_panel = validate_model_panel(
    pd.read_parquet(EXACT_PANEL_PATH),
    panel_name="Exact-run panel",
    expected_shape=EXPECTED_EXACT_SHAPE,
)

print("D2 shape:", d2_panel.shape)
print("Exact shape:", exact_panel.shape)

In [ ]:
# ============================================================
# 5. SAVE INPUT DATASET MANIFEST
# ============================================================

def panel_manifest(
    frame: pd.DataFrame,
    path: Path,
) -> dict:
    return {
        "path": str(path),
        "shape": list(frame.shape),
        "sha256": sha256_file(path),
        "first_timestamp_utc": str(frame["timestamp_utc"].min()),
        "last_timestamp_utc": str(frame["timestamp_utc"].max()),
        "price_areas": sorted(
            frame["price_area"].dropna().astype(str).unique().tolist()
        ),
        "target_column": TARGET_COLUMN,
    }


dataset_manifest = {
    "d2": panel_manifest(d2_panel, D2_PANEL_PATH),
    "exact": panel_manifest(exact_panel, EXACT_PANEL_PATH),
}

manifest_path = (
    AUDIT_DIR
    / "notebook_02_input_dataset_manifest.json"
)

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(dataset_manifest, file, indent=2)

display(
    pd.DataFrame(
        [
            {
                "panel": name,
                "rows": values["shape"][0],
                "columns": values["shape"][1],
                "first_timestamp_utc": values["first_timestamp_utc"],
                "last_timestamp_utc": values["last_timestamp_utc"],
                "sha256": values["sha256"][:16] + "...",
            }
            for name, values in dataset_manifest.items()
        ]
    )
)

## Decision-time feature registry

Every column receives one of the following statuses:

- `allowed`: eligible model input.
- `identifier`: key or descriptive field, not a predictor.
- `target`: realized wind-generation target.
- `benchmark_only`: retained for comparison but excluded from provider experts.
- `post_outcome`: only known after delivery or settlement.
- `manual_review`: ambiguous availability or semantics.

The automatic rules are conservative. Any `manual_review` item must be resolved explicitly before `RUN_MODE = "FULL"`.

In [ ]:
# ============================================================
# 6. COLUMN CLASSIFICATION UTILITIES
# ============================================================

IDENTIFIER_COLUMNS = {
    "timestamp_utc",
    "price_area",
    "target_date_local",
    "date_local",
    "local_date",
    "delivery_date",
    "model_alias",
    "api_model",
    "provider",
}

TARGET_PATTERNS = (
    r"^actual_wind",
    r"realized_wind",
    r"observed_wind",
)

BENCHMARK_PATTERNS = (
    r"energinet.*forecast",
    r"public.*forecast",
)

POST_OUTCOME_PATTERNS = (
    r"imbalance",
    r"regulating",
    r"settlement",
    r"realized_price",
    r"forecast_error",
    r"residual",
    r"actual_",
)

PRICE_PATTERNS = (
    r"day_ahead",
    r"elspot",
    r"spot_price",
    r"price_eur",
    r"price_dkk",
)

SAFE_SHARED_PATTERNS = (
    r"hour_",
    r"day_of_week",
    r"month",
    r"is_weekend",
    r"sin",
    r"cos",
    r"lag_",
    r"installed_capacity",
)

WEATHER_PATTERNS = (
    r"wind_speed",
    r"wind_direction",
    r"wind_gust",
    r"temperature",
    r"pressure",
    r"cloud",
    r"precip",
    r"humidity",
    r"cape",
    r"radiation",
)


def matches_any(
    value: str,
    patterns: Sequence[str],
) -> bool:
    return any(
        re.search(pattern, value, flags=re.IGNORECASE)
        for pattern in patterns
    )


def infer_provider(
    column: str,
    providers: Sequence[str],
) -> str:
    normalized = column.lower()

    for provider in sorted(providers, key=len, reverse=True):
        if provider.lower() in normalized:
            return provider

    return ""


def classify_column(
    column: str,
    dtype: str,
    providers: Sequence[str],
) -> dict:
    lower = column.lower()
    provider = infer_provider(column, providers)

    if column in IDENTIFIER_COLUMNS:
        status = "identifier"
        reason = "Key or descriptive field."
        group = "identifier"
    elif column == TARGET_COLUMN or matches_any(lower, TARGET_PATTERNS):
        status = "target"
        reason = "Realized wind-generation outcome."
        group = "target"
    elif matches_any(lower, BENCHMARK_PATTERNS):
        status = "benchmark_only"
        reason = "Public benchmark retained for evaluation."
        group = "benchmark"
    elif matches_any(lower, POST_OUTCOME_PATTERNS):
        status = "post_outcome"
        reason = "Potentially known only after delivery or settlement."
        group = "post_outcome"
    elif matches_any(lower, PRICE_PATTERNS):
        status = "manual_review"
        reason = (
            "Price availability depends on the precise nomination decision time; "
            "excluded from wind experts by default."
        )
        group = "price"
    elif provider and matches_any(lower, WEATHER_PATTERNS):
        status = "allowed"
        reason = "Provider weather feature."
        group = "provider_weather"
    elif matches_any(lower, SAFE_SHARED_PATTERNS):
        status = "allowed"
        reason = "Calendar, capacity, or valid lag feature."
        group = "shared"
    elif pd.api.types.is_numeric_dtype(dtype):
        status = "manual_review"
        reason = "Numeric feature requires semantic review."
        group = "other_numeric"
    else:
        status = "identifier"
        reason = "Non-numeric field excluded from default model matrix."
        group = "other"

    return {
        "feature": column,
        "dtype": str(dtype),
        "group": group,
        "provider": provider,
        "decision_time_status": status,
        "reason": reason,
        "include_default": status == "allowed",
    }

In [ ]:
# ============================================================
# 7. BUILD FEATURE REGISTRIES
# ============================================================

def build_feature_registry(
    frame: pd.DataFrame,
    providers: Sequence[str],
    panel_name: str,
) -> pd.DataFrame:
    registry = pd.DataFrame(
        [
            classify_column(
                column,
                frame[column].dtype,
                providers,
            )
            for column in frame.columns
        ]
    )

    registry.insert(0, "panel", panel_name)
    return registry


d2_feature_registry = build_feature_registry(
    d2_panel,
    D2_SOURCES,
    "d2",
)

exact_feature_registry = build_feature_registry(
    exact_panel,
    EXACT_SOURCES,
    "exact",
)

feature_registry = pd.concat(
    [d2_feature_registry, exact_feature_registry],
    ignore_index=True,
)

feature_registry.to_csv(
    AUDIT_DIR / "notebook_02_feature_registry.csv",
    index=False,
)

display(
    feature_registry.groupby(
        ["panel", "decision_time_status", "group"],
        as_index=False,
    ).size()
)

manual_review_features = feature_registry.loc[
    feature_registry["decision_time_status"].eq("manual_review")
].copy()

print("Manual-review feature count:", len(manual_review_features))
display(manual_review_features.head(100))

### Manual override cell

Use the dictionaries below to resolve any ambiguous columns after reviewing their semantics and publication time. Do not mark a feature `allowed` merely because it is correlated with the target.

Examples:

```python
D2_FEATURE_OVERRIDES = {
    "some_known_feature": "allowed",
    "some_realized_feature": "post_outcome",
}
```

In [ ]:
# ============================================================
# 8. MANUAL FEATURE-AVAILABILITY OVERRIDES
# ============================================================

D2_FEATURE_OVERRIDES: dict[str, str] = {
    # "feature_name": "allowed",
}

EXACT_FEATURE_OVERRIDES: dict[str, str] = {
    # "feature_name": "allowed",
}


def apply_feature_overrides(
    registry: pd.DataFrame,
    overrides: dict[str, str],
) -> pd.DataFrame:
    updated = registry.copy()

    valid_statuses = {
        "allowed",
        "identifier",
        "target",
        "benchmark_only",
        "post_outcome",
        "manual_review",
    }

    invalid = sorted(set(overrides.values()).difference(valid_statuses))
    if invalid:
        raise ValueError(f"Invalid override statuses: {invalid}")

    for feature, status in overrides.items():
        mask = updated["feature"].eq(feature)

        if not mask.any():
            raise KeyError(
                f"Override feature {feature!r} was not found."
            )

        updated.loc[mask, "decision_time_status"] = status
        updated.loc[mask, "include_default"] = status == "allowed"
        updated.loc[mask, "reason"] = "Manual Notebook 02 override."

    return updated


d2_feature_registry = apply_feature_overrides(
    d2_feature_registry,
    D2_FEATURE_OVERRIDES,
)

exact_feature_registry = apply_feature_overrides(
    exact_feature_registry,
    EXACT_FEATURE_OVERRIDES,
)

## Time-aware folds

The fold builder uses local delivery dates and expanding training windows. It never shuffles rows.

Each fold contains:

- an expanding training period;
- a contiguous validation period;
- a contiguous test period;
- no overlap among train, validation, and test dates.

The exact-run sample is shorter, so its fold lengths are deliberately smaller.

In [ ]:
# ============================================================
# 9. EXPANDING FOLD DEFINITIONS
# ============================================================

@dataclass(frozen=True)
class TemporalFold:
    panel: str
    fold_id: int
    train_start: str
    train_end: str
    validation_start: str
    validation_end: str
    test_start: str
    test_end: str


def local_dates(frame: pd.DataFrame) -> pd.Series:
    return (
        frame["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
        .dt.normalize()
    )


def make_expanding_folds(
    frame: pd.DataFrame,
    *,
    panel_name: str,
    min_train_days: int,
    validation_days: int,
    test_days: int,
    step_days: int,
) -> list[TemporalFold]:
    unique_dates = pd.DatetimeIndex(
        local_dates(frame).drop_duplicates().sort_values()
    )

    folds: list[TemporalFold] = []
    fold_id = 1
    cursor = min_train_days

    while (
        cursor + validation_days + test_days
        <= len(unique_dates)
    ):
        train_dates = unique_dates[:cursor]
        validation_slice = unique_dates[
            cursor : cursor + validation_days
        ]
        test_slice = unique_dates[
            cursor + validation_days :
            cursor + validation_days + test_days
        ]

        folds.append(
            TemporalFold(
                panel=panel_name,
                fold_id=fold_id,
                train_start=str(train_dates[0].date()),
                train_end=str(train_dates[-1].date()),
                validation_start=str(validation_slice[0].date()),
                validation_end=str(validation_slice[-1].date()),
                test_start=str(test_slice[0].date()),
                test_end=str(test_slice[-1].date()),
            )
        )

        cursor += step_days
        fold_id += 1

    if not folds:
        raise RuntimeError(
            f"No folds could be created for {panel_name}. "
            "Reduce fold lengths or inspect temporal support."
        )

    return folds


d2_folds = make_expanding_folds(
    d2_panel,
    panel_name="d2",
    **D2_FOLD_CONFIG,
)

exact_folds = make_expanding_folds(
    exact_panel,
    panel_name="exact",
    **EXACT_FOLD_CONFIG,
)

temporal_folds = pd.DataFrame(
    [asdict(fold) for fold in (*d2_folds, *exact_folds)]
)

temporal_folds.to_csv(
    AUDIT_DIR / "notebook_02_temporal_folds.csv",
    index=False,
)

display(temporal_folds)

In [ ]:
# ============================================================
# 10. FOLD MASKS AND INTEGRITY CHECKS
# ============================================================

def fold_masks(
    frame: pd.DataFrame,
    fold: TemporalFold,
) -> dict[str, pd.Series]:
    dates = local_dates(frame)

    train_start = pd.Timestamp(fold.train_start, tz=LOCAL_TIMEZONE)
    train_end = pd.Timestamp(fold.train_end, tz=LOCAL_TIMEZONE)

    validation_start = pd.Timestamp(
        fold.validation_start,
        tz=LOCAL_TIMEZONE,
    )
    validation_end = pd.Timestamp(
        fold.validation_end,
        tz=LOCAL_TIMEZONE,
    )

    test_start = pd.Timestamp(fold.test_start, tz=LOCAL_TIMEZONE)
    test_end = pd.Timestamp(fold.test_end, tz=LOCAL_TIMEZONE)

    masks = {
        "train": dates.between(train_start, train_end),
        "validation": dates.between(
            validation_start,
            validation_end,
        ),
        "test": dates.between(test_start, test_end),
    }

    overlap = (
        masks["train"].astype(int)
        + masks["validation"].astype(int)
        + masks["test"].astype(int)
    )

    if int((overlap > 1).sum()) != 0:
        raise RuntimeError(
            f"Fold {fold.panel}-{fold.fold_id} contains overlapping rows."
        )

    return masks


fold_integrity_rows = []

for panel_name, frame, folds in (
    ("d2", d2_panel, d2_folds),
    ("exact", exact_panel, exact_folds),
):
    for fold in folds:
        masks = fold_masks(frame, fold)

        for split_name, mask in masks.items():
            fold_integrity_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold.fold_id,
                    "split": split_name,
                    "rows": int(mask.sum()),
                    "unique_dates": int(
                        local_dates(frame.loc[mask]).nunique()
                    ),
                }
            )

fold_integrity = pd.DataFrame(fold_integrity_rows)
display(fold_integrity)

if (fold_integrity["rows"] == 0).any():
    raise RuntimeError("At least one fold split is empty.")

## Exploratory dependence analysis

The analysis below is performed on training periods only.

Required views:

- Pearson and Spearman feature-target correlation;
- provider-to-provider weather correlation;
- provider prediction residual correlation after the expert models exist;
- stability of correlation signs and ranks across folds;
- relationships by area, season, hour, wind regime, and provider-disagreement regime.

Correlation is descriptive, not a causal or automatic feature-selection rule.

In [ ]:
# ============================================================
# 11. TRAINING-ONLY FEATURE-TARGET CORRELATIONS
# ============================================================

def allowed_numeric_features(
    frame: pd.DataFrame,
    registry: pd.DataFrame,
) -> list[str]:
    allowed = registry.loc[
        registry["decision_time_status"].eq("allowed"),
        "feature",
    ].tolist()

    return [
        column
        for column in allowed
        if (
            column in frame.columns
            and pd.api.types.is_numeric_dtype(frame[column])
        )
    ]


def feature_target_correlations(
    frame: pd.DataFrame,
    registry: pd.DataFrame,
    folds: Sequence[TemporalFold],
    *,
    panel_name: str,
) -> pd.DataFrame:
    feature_columns = allowed_numeric_features(frame, registry)
    rows: list[dict] = []

    for fold in folds:
        masks = fold_masks(frame, fold)

        for price_area in PRICE_AREAS:
            area_train = frame.loc[
                masks["train"]
                & frame["price_area"].eq(price_area),
                [TARGET_COLUMN, *feature_columns],
            ]

            for feature in feature_columns:
                pair = area_train[[feature, TARGET_COLUMN]].dropna()

                if len(pair) < 50 or pair[feature].nunique() < 2:
                    continue

                rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold.fold_id,
                        "price_area": price_area,
                        "feature": feature,
                        "pearson": pair[feature].corr(
                            pair[TARGET_COLUMN],
                            method="pearson",
                        ),
                        "spearman": pair[feature].corr(
                            pair[TARGET_COLUMN],
                            method="spearman",
                        ),
                        "observations": len(pair),
                    }
                )

    return pd.DataFrame(rows)


d2_correlations = feature_target_correlations(
    d2_panel,
    d2_feature_registry,
    d2_folds,
    panel_name="d2",
)

exact_correlations = feature_target_correlations(
    exact_panel,
    exact_feature_registry,
    exact_folds,
    panel_name="exact",
)

feature_correlations = pd.concat(
    [d2_correlations, exact_correlations],
    ignore_index=True,
)

feature_correlations.to_parquet(
    AUDIT_DIR / "notebook_02_feature_target_correlations.parquet",
    index=False,
)

display(
    feature_correlations.assign(
        absolute_spearman=lambda frame: frame["spearman"].abs()
    )
    .sort_values(
        ["panel", "price_area", "absolute_spearman"],
        ascending=[True, True, False],
    )
    .groupby(["panel", "price_area"])
    .head(20)
)

In [ ]:
# ============================================================
# 12. CORRELATION STABILITY ACROSS FOLDS
# ============================================================

correlation_stability = (
    feature_correlations.groupby(
        ["panel", "price_area", "feature"],
        as_index=False,
    )
    .agg(
        mean_pearson=("pearson", "mean"),
        std_pearson=("pearson", "std"),
        mean_spearman=("spearman", "mean"),
        std_spearman=("spearman", "std"),
        min_spearman=("spearman", "min"),
        max_spearman=("spearman", "max"),
        folds=("fold_id", "nunique"),
    )
)

correlation_stability["sign_stable"] = (
    np.sign(correlation_stability["min_spearman"])
    == np.sign(correlation_stability["max_spearman"])
)

correlation_stability.to_csv(
    AUDIT_DIR / "notebook_02_correlation_stability.csv",
    index=False,
)

display(
    correlation_stability.sort_values(
        "mean_spearman",
        key=lambda values: values.abs(),
        ascending=False,
    ).head(50)
)

## Feature engineering

The default feature engineering is intentionally limited and interpretable:

- local calendar cycles;
- valid lagged wind production;
- wind-speed powers;
- direction sine/cosine conversion when raw degrees are present;
- provider disagreement and dispersion;
- cross-provider mean and median weather signals.

All lagged target features are built by timestamp alignment. They are not backfilled across missing periods.

In [ ]:
# ============================================================
# 13. SAFE FEATURE ENGINEERING
# ============================================================

def add_calendar_features(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    local_timestamp = (
        result["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )

    result["hour_local"] = local_timestamp.dt.hour
    result["day_of_week"] = local_timestamp.dt.dayofweek
    result["month"] = local_timestamp.dt.month
    result["day_of_year"] = local_timestamp.dt.dayofyear
    result["is_weekend"] = (
        local_timestamp.dt.dayofweek >= 5
    ).astype(int)

    result["hour_sin"] = np.sin(
        2 * np.pi * result["hour_local"] / 24
    )
    result["hour_cos"] = np.cos(
        2 * np.pi * result["hour_local"] / 24
    )
    result["day_of_year_sin"] = np.sin(
        2 * np.pi * result["day_of_year"] / 365.25
    )
    result["day_of_year_cos"] = np.cos(
        2 * np.pi * result["day_of_year"] / 365.25
    )

    return result


def add_target_lags(
    frame: pd.DataFrame,
    lag_hours: Sequence[int] = (24, 48, 168),
) -> pd.DataFrame:
    result = frame.copy()

    target_lookup = result[
        ["timestamp_utc", "price_area", TARGET_COLUMN]
    ].copy()

    for lag_hours_value in lag_hours:
        lagged = target_lookup.copy()
        lagged["timestamp_utc"] = (
            lagged["timestamp_utc"]
            + pd.Timedelta(hours=lag_hours_value)
        )
        lagged = lagged.rename(
            columns={
                TARGET_COLUMN: f"{TARGET_COLUMN}_lag_{lag_hours_value}h"
            }
        )

        result = result.merge(
            lagged,
            on=["timestamp_utc", "price_area"],
            how="left",
            validate="one_to_one",
        )

    return result


def add_wind_speed_powers(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    result = frame.copy()

    wind_speed_columns = [
        column
        for column in result.columns
        if (
            "wind_speed" in column.lower()
            and pd.api.types.is_numeric_dtype(result[column])
        )
    ]

    for column in wind_speed_columns:
        result[f"{column}__squared"] = result[column] ** 2
        result[f"{column}__cubed"] = result[column] ** 3

    return result


def add_provider_disagreement_features(
    frame: pd.DataFrame,
    providers: Sequence[str],
) -> pd.DataFrame:
    result = frame.copy()

    candidate_groups: dict[str, list[str]] = {}

    for column in result.columns:
        lower = column.lower()

        if "wind_speed" not in lower:
            continue

        matched_provider = infer_provider(column, providers)

        if not matched_provider:
            continue

        base_name = lower.replace(matched_provider.lower(), "{provider}")
        candidate_groups.setdefault(base_name, []).append(column)

    for base_name, columns in candidate_groups.items():
        if len(columns) < 2:
            continue

        safe_name = re.sub(r"[^a-zA-Z0-9]+", "_", base_name).strip("_")
        numeric_block = result[columns]

        result[f"provider_mean__{safe_name}"] = numeric_block.mean(axis=1)
        result[f"provider_median__{safe_name}"] = numeric_block.median(axis=1)
        result[f"provider_std__{safe_name}"] = numeric_block.std(axis=1)
        result[f"provider_range__{safe_name}"] = (
            numeric_block.max(axis=1)
            - numeric_block.min(axis=1)
        )

    return result


def engineer_features(
    frame: pd.DataFrame,
    providers: Sequence[str],
) -> pd.DataFrame:
    engineered = add_calendar_features(frame)
    engineered = add_target_lags(engineered)
    engineered = add_wind_speed_powers(engineered)
    engineered = add_provider_disagreement_features(
        engineered,
        providers,
    )

    return engineered


d2_engineered = engineer_features(d2_panel, D2_SOURCES)
exact_engineered = engineer_features(exact_panel, EXACT_SOURCES)

print("D2 engineered shape:", d2_engineered.shape)
print("Exact engineered shape:", exact_engineered.shape)

## Baselines

Notebook 02 reports simple, operationally meaningful baselines before fitting LightGBM:

- 24-hour persistence;
- 48-hour persistence;
- 168-hour seasonal persistence;
- expanding historical mean by area and local hour;
- Energinet public forecast when available.

Baselines are evaluated on the same test rows as the machine-learning models.

In [ ]:
# ============================================================
# 14. NUMERIC-SAFE METRICS AND BASELINE UTILITIES
# ============================================================

def _numeric_series(values: pd.Series | np.ndarray | list) -> pd.Series:
    """Return a float64 Series; invalid text becomes NaN."""
    series = pd.Series(values, copy=False)

    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").astype("float64")

    cleaned = (
        series.astype("string")
        .str.strip()
        .str.replace("\u00a0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(cleaned, errors="coerce").astype("float64")


def safe_regression_metrics(
    actual: pd.Series,
    predicted: pd.Series,
) -> dict[str, float]:
    # Convert positionally to avoid index-alignment surprises.
    actual_numeric = _numeric_series(actual).reset_index(drop=True)
    predicted_numeric = _numeric_series(predicted).reset_index(drop=True)

    valid = pd.DataFrame(
        {
            "actual": actual_numeric,
            "predicted": predicted_numeric,
        }
    ).replace(
        [np.inf, -np.inf],
        np.nan,
    ).dropna()

    if valid.empty:
        return {
            "mae": np.nan,
            "rmse": np.nan,
            "bias": np.nan,
            "nmae_mean": np.nan,
            "observations": 0,
        }

    actual_values = valid["actual"].to_numpy(dtype=float)
    predicted_values = valid["predicted"].to_numpy(dtype=float)

    error = predicted_values - actual_values
    mean_actual = float(np.mean(np.abs(actual_values)))

    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(np.sqrt(np.mean(error ** 2))),
        "bias": float(np.mean(error)),
        "nmae_mean": float(
            np.mean(np.abs(error)) / mean_actual
            if mean_actual > 0
            else np.nan
        ),
        "observations": int(len(valid)),
    }


def expanding_hourly_climatology(
    train: pd.DataFrame,
    evaluation: pd.DataFrame,
) -> pd.Series:
    train_local = (
        train["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )
    eval_local = (
        evaluation["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )

    training = train.assign(
        hour_local=train_local.dt.hour,
        month=train_local.dt.month,
    )

    lookup = (
        training.groupby(
            ["price_area", "month", "hour_local"]
        )[TARGET_COLUMN]
        .mean()
    )

    keys = pd.MultiIndex.from_arrays(
        [
            evaluation["price_area"],
            eval_local.dt.month,
            eval_local.dt.hour,
        ]
    )

    return pd.Series(
        lookup.reindex(keys).to_numpy(dtype=float),
        index=evaluation.index,
        name="climatology_prediction",
        dtype="float64",
    )

In [ ]:
# ============================================================
# 14A. VERIFY THAT THE NUMERIC-SAFE METRIC FUNCTION IS ACTIVE
# ============================================================

import inspect

metric_source = inspect.getsource(safe_regression_metrics)

assert "_numeric_series" in metric_source
assert "to_numpy(dtype=float)" in metric_source

print("Numeric-safe metric function is active.")
print(
    safe_regression_metrics(
        pd.Series([1.0, "2.0", "3,0"]),
        pd.Series(["1.5", 2.5, "4,0"]),
    )
)

In [ ]:
# ============================================================
# 15. GENERATE BASELINE PREDICTIONS FOR ALL FOLDS
# ============================================================

def baseline_predictions_for_panel(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    *,
    panel_name: str,
) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []

    lag_columns = {
        "persistence_24h": f"{TARGET_COLUMN}_lag_24h",
        "persistence_48h": f"{TARGET_COLUMN}_lag_48h",
        "persistence_168h": f"{TARGET_COLUMN}_lag_168h",
    }

    benchmark_candidates = [
        column
        for column in frame.columns
        if matches_any(column.lower(), BENCHMARK_PATTERNS)
    ]

    for fold in folds:
        masks = fold_masks(frame, fold)
        train = frame.loc[masks["train"]].copy()
        test = frame.loc[masks["test"]].copy()

        climatology = expanding_hourly_climatology(
            train,
            test,
        )

        prediction_map: dict[str, pd.Series] = {
            model_name: pd.to_numeric(
                test[column],
                errors="coerce",
            )
            for model_name, column in lag_columns.items()
            if column in test.columns
        }

        prediction_map["hourly_climatology"] = climatology

        for benchmark_column in benchmark_candidates:
            benchmark_numeric = pd.to_numeric(
                test[benchmark_column],
                errors="coerce",
            )

            # Skip columns that contain no usable numeric benchmark values.
            if benchmark_numeric.notna().any():
                prediction_map[
                    f"benchmark::{benchmark_column}"
                ] = benchmark_numeric

        for model_name, predictions in prediction_map.items():
            block = test[
                ["timestamp_utc", "price_area", TARGET_COLUMN]
            ].copy()

            block[TARGET_COLUMN] = pd.to_numeric(
                block[TARGET_COLUMN],
                errors="coerce",
            )

            block["panel"] = panel_name
            block["fold_id"] = fold.fold_id
            block["split"] = "test"
            block["model"] = model_name
            block["prediction"] = pd.to_numeric(
                pd.Series(
                    predictions,
                    index=test.index,
                ),
                errors="coerce",
            ).to_numpy(dtype=float)

            rows.append(block)

    return pd.concat(rows, ignore_index=True)


d2_baseline_predictions = baseline_predictions_for_panel(
    d2_engineered,
    d2_folds,
    panel_name="d2",
)

exact_baseline_predictions = baseline_predictions_for_panel(
    exact_engineered,
    exact_folds,
    panel_name="exact",
)

baseline_predictions = pd.concat(
    [d2_baseline_predictions, exact_baseline_predictions],
    ignore_index=True,
)

print(
    baseline_predictions.groupby(
        ["panel", "model"]
    )["prediction"]
    .agg(
        rows="size",
        numeric_values="count",
    )
)

display(baseline_predictions.head())

In [ ]:
# ============================================================
# 16. BASELINE METRIC TABLE — NUMERIC-SAFE
# ============================================================

# Sanitize once at the table boundary. This also fixes stale objects
# produced by an earlier execution of the baseline-generation cell.
baseline_predictions = baseline_predictions.copy()

baseline_predictions[TARGET_COLUMN] = _numeric_series(
    baseline_predictions[TARGET_COLUMN]
).to_numpy(dtype=float)

baseline_predictions["prediction"] = _numeric_series(
    baseline_predictions["prediction"]
).to_numpy(dtype=float)

print(
    "Metric input dtypes:",
    baseline_predictions[
        [TARGET_COLUMN, "prediction"]
    ].dtypes.to_dict(),
)

baseline_metric_rows = []

for keys, group in baseline_predictions.groupby(
    ["panel", "fold_id", "price_area", "model"]
):
    panel_name, fold_id, price_area, model_name = keys

    baseline_metric_rows.append(
        {
            "panel": panel_name,
            "fold_id": fold_id,
            "price_area": price_area,
            "model": model_name,
            **safe_regression_metrics(
                group[TARGET_COLUMN],
                group["prediction"],
            ),
        }
    )

baseline_metrics = pd.DataFrame(baseline_metric_rows)

baseline_metrics.to_csv(
    TABLE_DIR / "notebook_02_baseline_metrics.csv",
    index=False,
)

display(
    baseline_metrics.sort_values(
        ["panel", "price_area", "mae"]
    )
)

## Provider-specific LightGBM experts

Each provider expert receives:

- that provider's weather features;
- shared calendar and valid lag features;
- no other provider's weather fields;
- the same base model family and comparable tuning budget.

Hyperparameters may be tuned inside each training-validation fold, but no test observations may influence the selection.

In [ ]:
# ============================================================
# 17. OPTIONAL LIGHTGBM IMPORT
# ============================================================

try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor

    LIGHTGBM_AVAILABLE = True
    print("LightGBM version:", lgb.__version__)
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print(
        "LightGBM is not installed. "
        "Install it before RUN_MODE='FULL': py -m pip install lightgbm"
    )

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize

In [ ]:
# ============================================================
# 18. PROVIDER AND SHARED FEATURE SETS
# ============================================================

ENGINEERED_SHARED_FEATURES = (
    "hour_local",
    "day_of_week",
    "month",
    "day_of_year",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    f"{TARGET_COLUMN}_lag_24h",
    f"{TARGET_COLUMN}_lag_48h",
    f"{TARGET_COLUMN}_lag_168h",
)


def provider_feature_set(
    frame: pd.DataFrame,
    provider: str,
    registry: pd.DataFrame,
) -> list[str]:
    allowed_original = set(
        registry.loc[
            registry["decision_time_status"].eq("allowed"),
            "feature",
        ]
    )

    provider_columns = [
        column
        for column in frame.columns
        if (
            provider.lower() in column.lower()
            and pd.api.types.is_numeric_dtype(frame[column])
            and (
                column in allowed_original
                or "__squared" in column
                or "__cubed" in column
            )
        )
    ]

    shared_columns = [
        column
        for column in ENGINEERED_SHARED_FEATURES
        if (
            column in frame.columns
            and pd.api.types.is_numeric_dtype(frame[column])
        )
    ]

    return sorted(set(provider_columns + shared_columns))


def pooled_feature_set(
    frame: pd.DataFrame,
    registry: pd.DataFrame,
) -> list[str]:
    allowed_original = set(
        registry.loc[
            registry["decision_time_status"].eq("allowed"),
            "feature",
        ]
    )

    engineered_columns = [
        column
        for column in frame.columns
        if (
            "__squared" in column
            or "__cubed" in column
            or column.startswith("provider_")
            or column in ENGINEERED_SHARED_FEATURES
        )
    ]

    candidates = sorted(
        allowed_original.union(engineered_columns)
    )

    return [
        column
        for column in candidates
        if (
            column in frame.columns
            and pd.api.types.is_numeric_dtype(frame[column])
            and column != TARGET_COLUMN
        )
    ]


feature_set_summary = []

for panel_name, frame, providers, registry in (
    ("d2", d2_engineered, D2_SOURCES, d2_feature_registry),
    ("exact", exact_engineered, EXACT_SOURCES, exact_feature_registry),
):
    for provider in providers:
        features = provider_feature_set(
            frame,
            provider,
            registry,
        )

        feature_set_summary.append(
            {
                "panel": panel_name,
                "model": provider,
                "feature_count": len(features),
            }
        )

    feature_set_summary.append(
        {
            "panel": panel_name,
            "model": "pooled",
            "feature_count": len(
                pooled_feature_set(frame, registry)
            ),
        }
    )

display(pd.DataFrame(feature_set_summary))

In [ ]:
# ============================================================
# 19. LIGHTGBM FITTING FUNCTION
# ============================================================

def fit_lightgbm_regressor(
    train: pd.DataFrame,
    validation: pd.DataFrame,
    features: Sequence[str],
    *,
    params: dict | None = None,
):
    if not LIGHTGBM_AVAILABLE:
        raise RuntimeError(
            "LightGBM is unavailable. Install it before full modelling."
        )

    selected_params = dict(LIGHTGBM_BASE_PARAMS)
    if params:
        selected_params.update(params)

    model = LGBMRegressor(**selected_params)

    model.fit(
        train[list(features)],
        train[TARGET_COLUMN],
        eval_set=[
            (
                validation[list(features)],
                validation[TARGET_COLUMN],
            )
        ],
        eval_metric="mae",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False,
            ),
        ],
    )

    return model

In [ ]:
# ============================================================
# 20. PROVIDER-EXPERT TRAINING LOOP
# ============================================================

def train_provider_experts(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    providers: Sequence[str],
    registry: pd.DataFrame,
    *,
    panel_name: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prediction_blocks: list[pd.DataFrame] = []
    metadata_rows: list[dict] = []

    for fold in folds:
        masks = fold_masks(frame, fold)

        for price_area in PRICE_AREAS:
            area_mask = frame["price_area"].eq(price_area)

            train = frame.loc[
                masks["train"] & area_mask
            ].copy()

            validation = frame.loc[
                masks["validation"] & area_mask
            ].copy()

            test = frame.loc[
                masks["test"] & area_mask
            ].copy()

            for provider in providers:
                features = provider_feature_set(
                    frame,
                    provider,
                    registry,
                )

                if not features:
                    raise RuntimeError(
                        f"No features found for {panel_name}/{provider}."
                    )

                model = fit_lightgbm_regressor(
                    train,
                    validation,
                    features,
                )

                validation_prediction = model.predict(
                    validation[features]
                )
                test_prediction = model.predict(test[features])

                for split_name, split_frame, predictions in (
                    (
                        "validation",
                        validation,
                        validation_prediction,
                    ),
                    (
                        "test",
                        test,
                        test_prediction,
                    ),
                ):
                    block = split_frame[
                        [
                            "timestamp_utc",
                            "price_area",
                            TARGET_COLUMN,
                        ]
                    ].copy()

                    block["panel"] = panel_name
                    block["fold_id"] = fold.fold_id
                    block["split"] = split_name
                    block["model"] = provider
                    block["prediction"] = predictions

                    prediction_blocks.append(block)

                metadata_rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold.fold_id,
                        "price_area": price_area,
                        "provider": provider,
                        "feature_count": len(features),
                        "best_iteration": getattr(
                            model,
                            "best_iteration_",
                            None,
                        ),
                    }
                )

    return (
        pd.concat(prediction_blocks, ignore_index=True),
        pd.DataFrame(metadata_rows),
    )


if RUN_MODE == "FULL":
    d2_expert_predictions, d2_expert_metadata = (
        train_provider_experts(
            d2_engineered,
            d2_folds,
            D2_SOURCES,
            d2_feature_registry,
            panel_name="d2",
        )
    )

    exact_expert_predictions, exact_expert_metadata = (
        train_provider_experts(
            exact_engineered,
            exact_folds,
            EXACT_SOURCES,
            exact_feature_registry,
            panel_name="exact",
        )
    )

    expert_predictions = pd.concat(
        [d2_expert_predictions, exact_expert_predictions],
        ignore_index=True,
    )

    expert_metadata = pd.concat(
        [d2_expert_metadata, exact_expert_metadata],
        ignore_index=True,
    )
else:
    expert_predictions = pd.DataFrame()
    expert_metadata = pd.DataFrame()

    print(
        "AUDIT mode: provider-expert training was skipped."
    )

## Ensemble methods

The notebook evaluates:

- equal weighting;
- inverse validation-MAE weighting;
- constrained MAE weighting;
- constrained MSE weighting;
- best validation expert;
- linear stacking;
- dynamic exponentially weighted error weighting;
- pooled LightGBM.

All static weights are estimated from validation predictions only.

In [ ]:
# ============================================================
# 21. EXPERT-PREDICTION MATRIX
# ============================================================

def prediction_matrix(
    predictions: pd.DataFrame,
    *,
    split: str,
) -> pd.DataFrame:
    selected = predictions.loc[
        predictions["split"].eq(split)
    ].copy()

    wide = (
        selected.pivot_table(
            index=[
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
                TARGET_COLUMN,
            ],
            columns="model",
            values="prediction",
            aggfunc="first",
        )
        .reset_index()
    )

    wide.columns.name = None
    return wide


if not expert_predictions.empty:
    validation_matrix = prediction_matrix(
        expert_predictions,
        split="validation",
    )

    test_matrix = prediction_matrix(
        expert_predictions,
        split="test",
    )

    display(validation_matrix.head())
else:
    validation_matrix = pd.DataFrame()
    test_matrix = pd.DataFrame()

In [ ]:
# ============================================================
# 22. STATIC ENSEMBLE-WEIGHT ESTIMATORS
# ============================================================

def normalize_nonnegative_weights(
    weights: np.ndarray,
) -> np.ndarray:
    weights = np.clip(
        np.asarray(weights, dtype=float),
        0.0,
        None,
    )

    total = weights.sum()

    if total <= 0:
        return np.repeat(
            1.0 / len(weights),
            len(weights),
        )

    return weights / total


def optimize_simplex_weights(
    actual: np.ndarray,
    predictions: np.ndarray,
    *,
    loss: str,
) -> np.ndarray:
    n_models = predictions.shape[1]
    initial = np.repeat(1.0 / n_models, n_models)

    def objective(weights: np.ndarray) -> float:
        combined = predictions @ weights
        error = combined - actual

        if loss == "mae":
            return float(np.mean(np.abs(error)))
        if loss == "mse":
            return float(np.mean(error ** 2))

        raise ValueError(f"Unsupported loss: {loss}")

    result = minimize(
        objective,
        x0=initial,
        method="SLSQP",
        bounds=[(0.0, 1.0)] * n_models,
        constraints=[
            {
                "type": "eq",
                "fun": lambda weights: weights.sum() - 1.0,
            }
        ],
        options={
            "maxiter": 2_000,
            "ftol": 1e-10,
        },
    )

    if not result.success:
        warnings.warn(
            f"SLSQP did not converge: {result.message}"
        )

    return normalize_nonnegative_weights(result.x)


def inverse_mae_weights(
    actual: np.ndarray,
    predictions: np.ndarray,
    epsilon: float = 1e-9,
) -> np.ndarray:
    mae = np.mean(
        np.abs(predictions - actual[:, None]),
        axis=0,
    )

    return normalize_nonnegative_weights(
        1.0 / (mae + epsilon)
    )


def best_expert_weights(
    actual: np.ndarray,
    predictions: np.ndarray,
) -> np.ndarray:
    mae = np.mean(
        np.abs(predictions - actual[:, None]),
        axis=0,
    )

    weights = np.zeros(predictions.shape[1])
    weights[int(np.argmin(mae))] = 1.0
    return weights

In [ ]:
# ============================================================
# 23. FIT STATIC ENSEMBLE WEIGHTS
# ============================================================

def fit_ensemble_weights(
    validation_wide: pd.DataFrame,
    model_columns: Sequence[str],
) -> pd.DataFrame:
    rows: list[dict] = []

    for keys, group in validation_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        complete = group[
            [TARGET_COLUMN, *model_columns]
        ].dropna()

        actual = complete[TARGET_COLUMN].to_numpy()
        matrix = complete[list(model_columns)].to_numpy()

        methods = {
            "equal": np.repeat(
                1.0 / len(model_columns),
                len(model_columns),
            ),
            "inverse_mae": inverse_mae_weights(
                actual,
                matrix,
            ),
            "best_expert": best_expert_weights(
                actual,
                matrix,
            ),
            "slsqp_mae": optimize_simplex_weights(
                actual,
                matrix,
                loss="mae",
            ),
            "slsqp_mse": optimize_simplex_weights(
                actual,
                matrix,
                loss="mse",
            ),
        }

        for method, weights in methods.items():
            for model_name, weight in zip(
                model_columns,
                weights,
            ):
                rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold_id,
                        "price_area": price_area,
                        "ensemble_method": method,
                        "expert": model_name,
                        "weight": float(weight),
                    }
                )

    return pd.DataFrame(rows)


if not validation_matrix.empty:
    panel_provider_map = {
        "d2": list(D2_SOURCES),
        "exact": list(EXACT_SOURCES),
    }

    weight_blocks = []

    for panel_name, providers in panel_provider_map.items():
        panel_validation = validation_matrix.loc[
            validation_matrix["panel"].eq(panel_name)
        ]

        available_providers = [
            provider
            for provider in providers
            if provider in panel_validation.columns
        ]

        weight_blocks.append(
            fit_ensemble_weights(
                panel_validation,
                available_providers,
            )
        )

    ensemble_weights = pd.concat(
        weight_blocks,
        ignore_index=True,
    )
else:
    ensemble_weights = pd.DataFrame()

In [ ]:
# ============================================================
# 24. APPLY STATIC ENSEMBLES TO TEST PREDICTIONS
# ============================================================

def apply_static_ensembles(
    test_wide: pd.DataFrame,
    weights: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[pd.DataFrame] = []

    for keys, group in test_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        group_weights = weights.loc[
            weights["panel"].eq(panel_name)
            & weights["fold_id"].eq(fold_id)
            & weights["price_area"].eq(price_area)
        ]

        for method, method_weights in group_weights.groupby(
            "ensemble_method"
        ):
            experts = method_weights["expert"].tolist()
            weight_vector = method_weights.set_index("expert")[
                "weight"
            ].reindex(experts).to_numpy()

            complete_mask = group[experts].notna().all(axis=1)
            selected = group.loc[complete_mask].copy()

            prediction = (
                selected[experts].to_numpy()
                @ weight_vector
            )

            block = selected[
                [
                    "panel",
                    "fold_id",
                    "timestamp_utc",
                    "price_area",
                    TARGET_COLUMN,
                ]
            ].copy()

            block["split"] = "test"
            block["model"] = f"ensemble::{method}"
            block["prediction"] = prediction

            rows.append(block)

    return pd.concat(rows, ignore_index=True)


if not test_matrix.empty and not ensemble_weights.empty:
    static_ensemble_predictions = apply_static_ensembles(
        test_matrix,
        ensemble_weights,
    )
else:
    static_ensemble_predictions = pd.DataFrame()

In [ ]:
# ============================================================
# 25. LINEAR STACKING ON VALIDATION PREDICTIONS
# ============================================================

def fit_and_apply_linear_stack(
    validation_wide: pd.DataFrame,
    test_wide: pd.DataFrame,
    providers: Sequence[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    prediction_rows: list[pd.DataFrame] = []
    weight_rows: list[dict] = []

    for keys, validation_group in validation_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys

        experts = [
            provider
            for provider in providers
            if provider in validation_group.columns
        ]

        test_group = test_wide.loc[
            test_wide["panel"].eq(panel_name)
            & test_wide["fold_id"].eq(fold_id)
            & test_wide["price_area"].eq(price_area)
        ].copy()

        validation_complete = validation_group[
            [TARGET_COLUMN, *experts]
        ].dropna()

        test_complete = test_group.dropna(subset=experts)

        stacker = LinearRegression(
            fit_intercept=True,
            positive=True,
        )

        stacker.fit(
            validation_complete[experts],
            validation_complete[TARGET_COLUMN],
        )

        prediction = stacker.predict(
            test_complete[experts]
        )

        block = test_complete[
            [
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
                TARGET_COLUMN,
            ]
        ].copy()

        block["split"] = "test"
        block["model"] = "ensemble::linear_stack"
        block["prediction"] = prediction
        prediction_rows.append(block)

        for expert, coefficient in zip(
            experts,
            stacker.coef_,
        ):
            weight_rows.append(
                {
                    "panel": panel_name,
                    "fold_id": fold_id,
                    "price_area": price_area,
                    "ensemble_method": "linear_stack",
                    "expert": expert,
                    "weight": float(coefficient),
                    "intercept": float(stacker.intercept_),
                }
            )

    return (
        pd.concat(prediction_rows, ignore_index=True),
        pd.DataFrame(weight_rows),
    )


if not validation_matrix.empty:
    stack_prediction_blocks = []
    stack_weight_blocks = []

    for panel_name, providers in (
        ("d2", D2_SOURCES),
        ("exact", EXACT_SOURCES),
    ):
        predictions, weights = fit_and_apply_linear_stack(
            validation_matrix.loc[
                validation_matrix["panel"].eq(panel_name)
            ],
            test_matrix.loc[
                test_matrix["panel"].eq(panel_name)
            ],
            providers,
        )

        stack_prediction_blocks.append(predictions)
        stack_weight_blocks.append(weights)

    stacking_predictions = pd.concat(
        stack_prediction_blocks,
        ignore_index=True,
    )

    stacking_weights = pd.concat(
        stack_weight_blocks,
        ignore_index=True,
    )
else:
    stacking_predictions = pd.DataFrame()
    stacking_weights = pd.DataFrame()

## Dynamic weighting

The dynamic ensemble updates expert weights using only past absolute errors. The update is chronological within each test fold. No future test error is used to predict an earlier test hour.

The learning-rate parameter is frozen before final reporting.

In [ ]:
# ============================================================
# 26. HEDGE-STYLE DYNAMIC WEIGHTING
# ============================================================

HEDGE_LEARNING_RATE = 0.005


def hedge_predictions(
    wide: pd.DataFrame,
    experts: Sequence[str],
    *,
    learning_rate: float,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    ordered = wide.sort_values("timestamp_utc").copy()

    weights = np.repeat(
        1.0 / len(experts),
        len(experts),
    )

    prediction_rows = []
    weight_rows = []

    for _, row in ordered.iterrows():
        expert_values = row[list(experts)].to_numpy(dtype=float)

        if np.isnan(expert_values).any():
            continue

        ensemble_prediction = float(expert_values @ weights)

        prediction_rows.append(
            {
                "panel": row["panel"],
                "fold_id": row["fold_id"],
                "timestamp_utc": row["timestamp_utc"],
                "price_area": row["price_area"],
                TARGET_COLUMN: row[TARGET_COLUMN],
                "split": "test",
                "model": "ensemble::hedge",
                "prediction": ensemble_prediction,
            }
        )

        for expert, weight in zip(experts, weights):
            weight_rows.append(
                {
                    "panel": row["panel"],
                    "fold_id": row["fold_id"],
                    "timestamp_utc": row["timestamp_utc"],
                    "price_area": row["price_area"],
                    "expert": expert,
                    "weight": float(weight),
                }
            )

        realized_error = np.abs(
            expert_values - float(row[TARGET_COLUMN])
        )

        weights = normalize_nonnegative_weights(
            weights
            * np.exp(
                -learning_rate * realized_error
            )
        )

    return (
        pd.DataFrame(prediction_rows),
        pd.DataFrame(weight_rows),
    )


if not test_matrix.empty:
    hedge_prediction_blocks = []
    hedge_weight_blocks = []

    for keys, group in test_matrix.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys
        providers = (
            D2_SOURCES
            if panel_name == "d2"
            else EXACT_SOURCES
        )

        experts = [
            provider
            for provider in providers
            if provider in group.columns
        ]

        predictions, weights = hedge_predictions(
            group,
            experts,
            learning_rate=HEDGE_LEARNING_RATE,
        )

        hedge_prediction_blocks.append(predictions)
        hedge_weight_blocks.append(weights)

    hedge_prediction_frame = pd.concat(
        hedge_prediction_blocks,
        ignore_index=True,
    )

    hedge_weight_history = pd.concat(
        hedge_weight_blocks,
        ignore_index=True,
    )
else:
    hedge_prediction_frame = pd.DataFrame()
    hedge_weight_history = pd.DataFrame()

## Pooled LightGBM

The pooled model receives the complete allowed weather feature set from all providers plus shared engineered features. It is a benchmark against explicit expert combination, not a replacement for provider-level interpretability.

In [ ]:
# ============================================================
# 27. POOLED LIGHTGBM TRAINING
# ============================================================

def train_pooled_model(
    frame: pd.DataFrame,
    folds: Sequence[TemporalFold],
    registry: pd.DataFrame,
    *,
    panel_name: str,
) -> pd.DataFrame:
    prediction_blocks: list[pd.DataFrame] = []
    features = pooled_feature_set(frame, registry)

    for fold in folds:
        masks = fold_masks(frame, fold)

        for price_area in PRICE_AREAS:
            area_mask = frame["price_area"].eq(price_area)

            train = frame.loc[
                masks["train"] & area_mask
            ].copy()

            validation = frame.loc[
                masks["validation"] & area_mask
            ].copy()

            test = frame.loc[
                masks["test"] & area_mask
            ].copy()

            model = fit_lightgbm_regressor(
                train,
                validation,
                features,
            )

            prediction = model.predict(test[features])

            block = test[
                ["timestamp_utc", "price_area", TARGET_COLUMN]
            ].copy()

            block["panel"] = panel_name
            block["fold_id"] = fold.fold_id
            block["split"] = "test"
            block["model"] = "pooled_lightgbm"
            block["prediction"] = prediction

            prediction_blocks.append(block)

    return pd.concat(prediction_blocks, ignore_index=True)


if RUN_MODE == "FULL":
    d2_pooled_predictions = train_pooled_model(
        d2_engineered,
        d2_folds,
        d2_feature_registry,
        panel_name="d2",
    )

    exact_pooled_predictions = train_pooled_model(
        exact_engineered,
        exact_folds,
        exact_feature_registry,
        panel_name="exact",
    )

    pooled_predictions = pd.concat(
        [d2_pooled_predictions, exact_pooled_predictions],
        ignore_index=True,
    )
else:
    pooled_predictions = pd.DataFrame()

## Out-of-sample evaluation

Primary metrics:

- MAE;
- RMSE;
- bias;
- normalized MAE;
- daily aggregate absolute error;
- ramp error;
- performance by season, hour, wind regime, and provider-disagreement regime;
- fold-to-fold stability;
- expert-weight stability.

The test sets are the only rows used for final comparative reporting.

In [ ]:
# ============================================================
# 28. COMBINE ALL TEST PREDICTIONS
# ============================================================

prediction_frames = [
    baseline_predictions,
]

for optional_frame in (
    expert_predictions.loc[
        expert_predictions["split"].eq("test")
    ] if not expert_predictions.empty else pd.DataFrame(),
    static_ensemble_predictions,
    stacking_predictions,
    hedge_prediction_frame,
    pooled_predictions,
):
    if not optional_frame.empty:
        prediction_frames.append(optional_frame)

all_test_predictions = pd.concat(
    prediction_frames,
    ignore_index=True,
)

all_test_predictions.to_parquet(
    PREDICTION_DIR / "notebook_02_test_predictions.parquet",
    index=False,
)

print("Combined prediction rows:", len(all_test_predictions))
display(all_test_predictions.head())

In [ ]:
# ============================================================
# 29. PRIMARY ACCURACY TABLE
# ============================================================

metric_rows = []

for keys, group in all_test_predictions.groupby(
    ["panel", "fold_id", "price_area", "model"]
):
    panel_name, fold_id, price_area, model_name = keys

    metric_rows.append(
        {
            "panel": panel_name,
            "fold_id": fold_id,
            "price_area": price_area,
            "model": model_name,
            **safe_regression_metrics(
                group[TARGET_COLUMN],
                group["prediction"],
            ),
        }
    )

model_metrics = pd.DataFrame(metric_rows)

model_metrics.to_csv(
    TABLE_DIR / "notebook_02_model_metrics_by_fold.csv",
    index=False,
)

display(
    model_metrics.sort_values(
        ["panel", "price_area", "fold_id", "mae"]
    )
)

In [ ]:
# ============================================================
# 30. AGGREGATE ACCURACY AND STABILITY
# ============================================================

metric_summary = (
    model_metrics.groupby(
        ["panel", "price_area", "model"],
        as_index=False,
    )
    .agg(
        mean_mae=("mae", "mean"),
        std_mae=("mae", "std"),
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_bias=("bias", "mean"),
        mean_nmae=("nmae_mean", "mean"),
        folds=("fold_id", "nunique"),
        observations=("observations", "sum"),
    )
)

metric_summary["mae_cv"] = (
    metric_summary["std_mae"]
    / metric_summary["mean_mae"]
)

metric_summary.to_csv(
    TABLE_DIR / "notebook_02_model_metric_summary.csv",
    index=False,
)

display(
    metric_summary.sort_values(
        ["panel", "price_area", "mean_mae"]
    )
)

In [ ]:
# ============================================================
# 31. REGIME LABELS
# ============================================================

def add_evaluation_regimes(
    predictions: pd.DataFrame,
) -> pd.DataFrame:
    result = predictions.copy()

    local_timestamp = (
        result["timestamp_utc"]
        .dt.tz_convert(LOCAL_TIMEZONE)
    )

    result["hour_local"] = local_timestamp.dt.hour
    result["month"] = local_timestamp.dt.month

    result["season"] = pd.cut(
        result["month"],
        bins=[0, 2, 5, 8, 11, 12],
        labels=[
            "winter",
            "spring",
            "summer",
            "autumn",
            "winter_end",
        ],
        include_lowest=True,
    ).astype(str)

    result.loc[
        result["season"].eq("winter_end"),
        "season",
    ] = "winter"

    result["wind_regime"] = (
        result.groupby(
            ["panel", "price_area"]
        )[TARGET_COLUMN]
        .transform(
            lambda values: pd.qcut(
                values.rank(method="first"),
                q=3,
                labels=["low", "medium", "high"],
            )
        )
    )

    result["absolute_error"] = (
        result["prediction"]
        - result[TARGET_COLUMN]
    ).abs()

    return result


regime_predictions = add_evaluation_regimes(
    all_test_predictions
)

In [ ]:
# ============================================================
# 32. REGIME PERFORMANCE TABLE
# ============================================================

regime_metric_rows = []

for regime_name in (
    "season",
    "hour_local",
    "wind_regime",
):
    for keys, group in regime_predictions.groupby(
        [
            "panel",
            "price_area",
            "model",
            regime_name,
        ],
        dropna=False,
    ):
        panel_name, price_area, model_name, regime_value = keys

        regime_metric_rows.append(
            {
                "panel": panel_name,
                "price_area": price_area,
                "model": model_name,
                "regime_type": regime_name,
                "regime_value": str(regime_value),
                **safe_regression_metrics(
                    group[TARGET_COLUMN],
                    group["prediction"],
                ),
            }
        )

regime_metrics = pd.DataFrame(regime_metric_rows)

regime_metrics.to_csv(
    TABLE_DIR / "notebook_02_regime_metrics.csv",
    index=False,
)

display(regime_metrics.head(50))

In [ ]:
# ============================================================
# 33. RESIDUAL CORRELATION AMONG PROVIDER EXPERTS
# ============================================================

if not expert_predictions.empty:
    expert_test = expert_predictions.loc[
        expert_predictions["split"].eq("test")
    ].copy()

    expert_test["residual"] = (
        expert_test["prediction"]
        - expert_test[TARGET_COLUMN]
    )

    residual_wide = (
        expert_test.pivot_table(
            index=[
                "panel",
                "fold_id",
                "timestamp_utc",
                "price_area",
            ],
            columns="model",
            values="residual",
            aggfunc="first",
        )
        .reset_index()
    )

    residual_correlation_rows = []

    for keys, group in residual_wide.groupby(
        ["panel", "fold_id", "price_area"]
    ):
        panel_name, fold_id, price_area = keys
        numeric = group.select_dtypes(include=[np.number])
        correlation = numeric.corr()

        for expert_a in correlation.columns:
            for expert_b in correlation.columns:
                if expert_a >= expert_b:
                    continue

                residual_correlation_rows.append(
                    {
                        "panel": panel_name,
                        "fold_id": fold_id,
                        "price_area": price_area,
                        "expert_a": expert_a,
                        "expert_b": expert_b,
                        "residual_correlation": correlation.loc[
                            expert_a,
                            expert_b,
                        ],
                    }
                )

    residual_correlations = pd.DataFrame(
        residual_correlation_rows
    )

    residual_correlations.to_csv(
        AUDIT_DIR
        / "notebook_02_expert_residual_correlations.csv",
        index=False,
    )

    display(
        residual_correlations.sort_values(
            "residual_correlation"
        ).head(50)
    )
else:
    residual_correlations = pd.DataFrame()
    print(
        "Residual-correlation analysis requires FULL mode."
    )

In [ ]:
# ============================================================
# 34. ENSEMBLE-WEIGHT STABILITY
# ============================================================

weight_frames = []

if not ensemble_weights.empty:
    weight_frames.append(
        ensemble_weights.assign(
            weight_source="static"
        )
    )

if not stacking_weights.empty:
    weight_frames.append(
        stacking_weights.assign(
            weight_source="stacking"
        )
    )

if weight_frames:
    combined_weights = pd.concat(
        weight_frames,
        ignore_index=True,
        sort=False,
    )

    weight_stability = (
        combined_weights.groupby(
            [
                "panel",
                "price_area",
                "ensemble_method",
                "expert",
            ],
            as_index=False,
        )
        .agg(
            mean_weight=("weight", "mean"),
            std_weight=("weight", "std"),
            min_weight=("weight", "min"),
            max_weight=("weight", "max"),
            folds=("fold_id", "nunique"),
        )
    )

    weight_stability.to_csv(
        TABLE_DIR / "notebook_02_weight_stability.csv",
        index=False,
    )

    display(weight_stability)
else:
    combined_weights = pd.DataFrame()
    weight_stability = pd.DataFrame()
    print("Weight-stability analysis requires FULL mode.")

## Exact-run and D2 comparison

The two tracks are not concatenated. Comparisons are restricted to overlapping timestamps and use separately trained models.

Useful comparisons:

- best D2 expert versus best exact-run expert;
- pooled D2 versus pooled exact-run;
- equal-weight D2 versus equal-weight exact-run;
- coverage and sample-size trade-offs;
- whether exact information-time realism changes provider ranking.

In [ ]:
# ============================================================
# 35. OVERLAPPING-PERIOD COMPARISON SHELL
# ============================================================

def overlapping_prediction_comparison(
    predictions: pd.DataFrame,
    model_pairs: Sequence[tuple[str, str]],
) -> pd.DataFrame:
    rows = []

    for d2_model, exact_model in model_pairs:
        d2_selected = predictions.loc[
            predictions["panel"].eq("d2")
            & predictions["model"].eq(d2_model)
        ].copy()

        exact_selected = predictions.loc[
            predictions["panel"].eq("exact")
            & predictions["model"].eq(exact_model)
        ].copy()

        overlap = d2_selected.merge(
            exact_selected,
            on=["timestamp_utc", "price_area"],
            how="inner",
            suffixes=("_d2", "_exact"),
        )

        for track, target_col, prediction_col in (
            (
                "d2",
                f"{TARGET_COLUMN}_d2",
                "prediction_d2",
            ),
            (
                "exact",
                f"{TARGET_COLUMN}_exact",
                "prediction_exact",
            ),
        ):
            metrics = safe_regression_metrics(
                overlap[target_col],
                overlap[prediction_col],
            )

            rows.append(
                {
                    "d2_model": d2_model,
                    "exact_model": exact_model,
                    "evaluated_track": track,
                    **metrics,
                }
            )

    return pd.DataFrame(rows)


OVERLAP_MODEL_PAIRS = (
    ("hourly_climatology", "hourly_climatology"),
    ("ensemble::equal", "ensemble::equal"),
    ("pooled_lightgbm", "pooled_lightgbm"),
)

if RUN_MODE == "FULL":
    overlap_comparison = overlapping_prediction_comparison(
        all_test_predictions,
        OVERLAP_MODEL_PAIRS,
    )

    overlap_comparison.to_csv(
        TABLE_DIR / "notebook_02_d2_exact_overlap_comparison.csv",
        index=False,
    )

    display(overlap_comparison)
else:
    overlap_comparison = pd.DataFrame()

## Verification gate

Expected audit status:

- `READY_FOR_FULL_MODELLING` when the frozen-data, feature-registry, fold, and baseline gates pass in `AUDIT` mode.
- `MODELLING_COMPLETE` when full expert, ensemble, pooled-model, and evaluation outputs are available in `FULL` mode.

A non-empty unresolved `manual_review` feature set is a blocking failure for `FULL` mode.

In [ ]:
# ============================================================
# 36. FINAL VERIFICATION
# ============================================================

blocking_failures: list[str] = []

if d2_panel.shape != EXPECTED_D2_SHAPE:
    blocking_failures.append(
        f"D2 shape mismatch: {d2_panel.shape}"
    )

if exact_panel.shape != EXPECTED_EXACT_SHAPE:
    blocking_failures.append(
        f"Exact shape mismatch: {exact_panel.shape}"
    )

if temporal_folds.empty:
    blocking_failures.append(
        "Temporal fold manifest is empty."
    )

remaining_manual_review = pd.concat(
    [
        d2_feature_registry.loc[
            d2_feature_registry[
                "decision_time_status"
            ].eq("manual_review")
        ],
        exact_feature_registry.loc[
            exact_feature_registry[
                "decision_time_status"
            ].eq("manual_review")
        ],
    ],
    ignore_index=True,
)

if RUN_MODE == "FULL" and not remaining_manual_review.empty:
    blocking_failures.append(
        "Manual-review features remain unresolved."
    )

if RUN_MODE == "FULL":
    required_full_outputs = {
        "expert_predictions": expert_predictions,
        "ensemble_weights": ensemble_weights,
        "pooled_predictions": pooled_predictions,
        "model_metrics": model_metrics,
    }

    for output_name, output_frame in required_full_outputs.items():
        if output_frame.empty:
            blocking_failures.append(
                f"Full-mode output is empty: {output_name}"
            )

if blocking_failures:
    verification_status = "BLOCKED"
elif RUN_MODE == "AUDIT":
    verification_status = "READY_FOR_FULL_MODELLING"
else:
    verification_status = "MODELLING_COMPLETE"

verification = {
    "verification_status": verification_status,
    "run_mode": RUN_MODE,
    "blocking_failures": blocking_failures,
    "d2_shape": list(d2_panel.shape),
    "exact_shape": list(exact_panel.shape),
    "d2_folds": len(d2_folds),
    "exact_folds": len(exact_folds),
    "manual_review_feature_count": int(
        len(remaining_manual_review)
    ),
    "baseline_prediction_rows": int(
        len(baseline_predictions)
    ),
    "all_test_prediction_rows": int(
        len(all_test_predictions)
    ),
    "input_manifest": dataset_manifest,
}

verification_path = (
    AUDIT_DIR
    / "notebook_02_verification.json"
)

with verification_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        verification,
        file,
        indent=2,
        default=str,
    )

print("=== NOTEBOOK 02 VERIFICATION ===")
print(
    json.dumps(
        verification,
        indent=2,
        default=str,
    )
)

## Outputs

### Audit outputs

- `outputs/audits/notebook_02_input_dataset_manifest.json`
- `outputs/audits/notebook_02_feature_registry.csv`
- `outputs/audits/notebook_02_temporal_folds.csv`
- `outputs/audits/notebook_02_feature_target_correlations.parquet`
- `outputs/audits/notebook_02_correlation_stability.csv`
- `outputs/audits/notebook_02_expert_residual_correlations.csv`
- `outputs/audits/notebook_02_verification.json`

### Tables

- `outputs/tables/notebook_02_baseline_metrics.csv`
- `outputs/tables/notebook_02_model_metrics_by_fold.csv`
- `outputs/tables/notebook_02_model_metric_summary.csv`
- `outputs/tables/notebook_02_regime_metrics.csv`
- `outputs/tables/notebook_02_weight_stability.csv`
- `outputs/tables/notebook_02_d2_exact_overlap_comparison.csv`

### Predictions

- `outputs/predictions/notebook_02_test_predictions.parquet`

Prediction and model binaries should remain excluded from Git. Compact audit summaries, selected tables, and publication figures may be committed after review.